# Data Preprocessing
In this Notebook we perform data cleaning and Preparation for Multi-Omics Analysis

#### Import Packages

In [ ]:
# ============================================
# Preprocessing Notebook — Setup
# ============================================
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

# --- Core ---
import re
import warnings
from io import BytesIO
import tarfile
import gzip

import numpy as np
import pandas as pd

# --- Optional file readers (only needed if you actually read .xls) ---
import xlrd  # for legacy .xls files

# --- ML / Imputation ---
from sklearn.impute import KNNImputer

# --- Project utils ---
from modules.utils import save_json, load_json

# --- Global settings ---
warnings.filterwarnings("ignore")


# ============================================
# Standardization Maps (to unify cohort terms)
# ============================================

# Use these mappings to ensure consistent labels across datasets/cohorts.
word_map = {
    "Gender": {
        "male": "Male", "female": "Female",
        "Male": "Male", "Female": "Female",
        "M": "Male", "F": "Female",
        "0.0": "Female", "1.0": "Male",
    },
    "Diagnosis": {
        "AD": "AD",
        "ad": "AD",
        "Alzheimer Disease": "AD",
        "Alzheimer's Disease": "AD",
        "alzheimer patient": "AD",

        "normal": "Control",
        "control": "Control",
        "NC": "Control",
        "Health": "Control",
        "Healthy Control": "Control",
        "Normal Aging": "Control",
        "Normal Young": "Control",
        "Normal Old": "Control",
        "Sham": "Control",

        "MCI": "MCI",
        "mild cognitive impairment (MCI)": "MCI",
        "Mild cognitive impairment due to Alzheimer's disease": "MCI",

        "DLB": "DLB",
        "NPH": "NPH",
        "VaD": "VaD",
        "VCID": "VCID",
        "NVCID": "NVCID",
        "HY": "HY",
        "PD": "PD",
        "pd": "PD",

        "Incipient": "Incipient AD",
        "Moderate": "Moderate AD",
        "Severe": "Severe AD",
        "early stage (control)": "Early Stage AD",
        "late stage": "Late Stage AD",

        # numeric encodings seen in some cohorts
        "1.0": "Control",
        "2.0": "MCI",
        "3.0": "MCI+",
        "4.0": "AD",
        "5.0": "AD+",
        "6.0": "Other",

        # keep as-is (if you truly want this exact label)
        "MCI,30": "MCI,30",
    },
}


# Make cohort directory for saving clean data
os.makedirs("../data/ROSMAP", exist_ok=True)

#### 1. Preprocess MicroRNA Data

In [ ]:
# load processed data
expression_data = pd.read_csv("../downloads/ROSMAP/ROSMAP_arraymiRNA.gct", sep="\t", skiprows =1)
expression_data.columns = expression_data.iloc[0,:]
expression_data = expression_data[1:]
expression_data.index = expression_data.Genes
expression_data.drop(["Genes", "Description"], axis=1, inplace=True)
expression_data = expression_data.T
expression_data.head()

In [ ]:
# load metadata
metadata = pd.read_csv("../downloads/ROSMAP/ROSMAP_assay_miRNAarray_nanostring_metadata.csv")
metadata.head()

In [ ]:
# load clinical data
clinical_data = pd.read_csv("../downloads/ROSMAP/ROSMAP_clinical.csv")
clinical_data.head()

In [ ]:
# load ROSMAP_biospecimen_metadata
biospecimen_metadata = pd.read_csv("../downloads/ROSMAP/ROSMAP_biospecimen_metadata.csv")
biospecimen_metadata.head()

In [ ]:
# create specimen_to_individual_id map
df = biospecimen_metadata.iloc[:,0:2]
df.dropna(inplace=True)
specimenID_to_individualID_map = dict(zip(df.specimenID, df.individualID))

In [ ]:
# create sample_id to specimen id map
df2 = metadata.iloc[:,0:2]
df2.dropna(inplace = True)
sampleID_to_specimenID_map = dict(zip(df2.mirna_id, df2.specimenID))

In [ ]:
# How many known sample ids are there
print("Number of known samples: ", len(set(expression_data.index)))

In [ ]:
# How many samples have mapped speciment ids
mapped_samples = [sample_id for sample_id in expression_data.index if sample_id in sampleID_to_specimenID_map]

print("Number of mapped sample_ids to specimen ids: ", len(set(mapped_samples)))

In [ ]:
# How many mapped specimen ids have mapped individual ids?
mapped_specimens = [sampleID_to_specimenID_map[sample_id] for sample_id in mapped_samples \
                    if sampleID_to_specimenID_map[sample_id] in specimenID_to_individualID_map]

print("Number of sample ids with individual id mappings: ", len(mapped_specimens))

In [ ]:
# fecth the sample ids that have individualid mapping
mappable_sampleIDs = [sample_id for sample_id in expression_data.index \
                     if sample_id in mapped_samples and sampleID_to_specimenID_map[sample_id] in mapped_specimens
                     ]

len(mappable_sampleIDs)

In [ ]:
# map expression data indices
expression_data = expression_data.loc[mappable_sampleIDs]
expression_data.index = [specimenID_to_individualID_map[sampleID_to_specimenID_map[sample_id]] \
                        for sample_id in expression_data.index]
expression_data.head()

In [ ]:
# filter clinical data by common individual ids
clinical_data.index = clinical_data.individualID
clinical_data = clinical_data.loc[expression_data.index]
clinical_data.head()

**Now let's clean the individual data**  
Columns to select:
* msex: Gender
* race: Race
* pmi: Post-mortem interval
* braaksc: Braak Stage
* cogdx:  Diagnosis

In [ ]:
# clean clinical data
clinical_data = clinical_data[["msex", "race", "pmi", "braaksc", "cogdx"]]
clinical_data.columns = ["Gender", "Race", "PMI", "Braak", "Diagnosis"]

clinical_data["Race"] = clinical_data["Race"].astype(int)
clinical_data["Gender"] = clinical_data["Gender"].apply(lambda x: word_map["Gender"][str(x)])
clinical_data["Diagnosis"] = clinical_data["Diagnosis"].apply(lambda x: word_map["Diagnosis"][str(x)])
clinical_data["Braak"] = clinical_data["Braak"].astype(int)
clinical_data.head()

In [ ]:
# now merge clinical and expression data
microRNA_data = pd.merge(clinical_data.reset_index(), expression_data.reset_index())
microRNA_data.set_index("index", inplace = True)
microRNA_data.index.rename('Sample ID', inplace=True)

# display data
microRNA_data.head()

#### 2. Preprocess Gene Expression Data

In [16]:
# load metadata
rnaseq_met = pd.read_csv("../downloads/ROSMAP/ROSMAP_biospecimen_metadata.csv")
rnaseq_met.head()

,individualID,specimenID,specimenIdSource,organ,tissue,BrodmannArea,sampleStatus,tissueWeight,tissueVolume,nucleicAcidSource,cellType,fastingState,isPostMortem,samplingAge,samplingAgeUnits,visitNumber,assay,exclude,excludeReason,samplingDate
0,R1743384,190403-B4-A_R1743384,NaN,brain,dorsolateral prefrontal cortex,NaN,frozen,NaN,NaN,single nucleus,NaN,NaN,True,NaN,NaN,NaN,scrnaSeq,False,NaN,NaN
1,R2670295,190403-B4-A_R2670295,NaN,brain,dorsolateral prefrontal cortex,NaN,frozen,NaN,NaN,single nucleus,NaN,NaN,True,NaN,NaN,NaN,scrnaSeq,False,NaN,NaN
2,R4119160,190403-B4-A_R4119160,NaN,brain,dorsolateral prefrontal cortex,NaN,frozen,NaN,NaN,single nucleus,NaN,NaN,True,NaN,NaN,NaN,scrnaSeq,True,RNA genotype discordant with WGS,NaN
3,R4641987,190403-B4-A_R4641987,NaN,brain,dorsolateral prefrontal cortex,NaN,frozen,NaN,NaN,single nucleus,NaN,NaN,True,NaN,NaN,NaN,scrnaSeq,False,NaN,NaN
4,R5693901,190403-B4-A_R5693901,NaN,brain,dorsolateral prefrontal cortex,NaN,frozen,NaN,NaN,single nucleus,NaN,NaN,True,NaN,NaN,NaN,scrnaSeq,True,Duplicated donor,NaN


In [17]:
# Load gene expression data
gene_expression = pd.read_csv("../downloads/ROSMAP/ROSMAP_RNAseq_FPKM_gene.tsv", sep = "\t")
rnaseq_sample_names = gene_expression.columns[2:]
gene_expression = gene_expression.iloc[:,1:]
gene_expression.index = gene_expression.gene_id
gene_expression = gene_expression.drop("gene_id", axis=1).T
gene_expression.head()

gene_id,ENSG00000167578.11,ENSG00000242268.1,ENSG00000078237.4,ENSG00000263642.1,ENSG00000225275.4,ENSG00000060642.6,ENSG00000201788.1,ENSG00000263089.1,ENSG00000172137.14,ENSG00000240423.1,...,ENSG00000232668.1,ENSG00000089177.13,ENSG00000216352.1,ENSG00000267117.1,ENSG00000148943.7,ENSG00000265520.1,ENSG00000231119.2,ENSG00000105063.14,ENSG00000123685.4,ENSG00000181518.2
525_120515_0,60.84,0.08,4.39,0.0,0.0,5.98,0.0,0.0,19.69,0.09,...,0.0,4.98,0.0,0.0,10.02,0.0,0.20,40.59,4.46,0.00
383_120503_0,65.45,0.05,4.49,0.0,0.0,4.66,0.0,0.0,31.08,0.18,...,0.0,5.33,0.0,0.0,8.76,0.0,0.20,39.26,4.51,0.00
93_120417_0,69.18,0.08,2.51,0.0,0.0,2.77,0.0,0.0,10.76,0.12,...,0.0,6.95,0.0,0.0,6.28,0.0,0.24,34.70,5.27,0.05
610_120523_0,51.60,0.08,2.90,0.0,0.0,4.50,0.0,0.0,9.14,0.18,...,0.0,5.31,0.0,0.0,8.32,0.0,0.21,36.74,5.11,0.05
560_120517_0,48.61,0.10,2.67,0.0,0.0,4.26,0.0,0.0,5.87,0.17,...,0.0,7.12,0.0,0.0,8.68,0.0,0.09,39.85,3.77,0.00


In [18]:
# Sample ID names extensions
set([x.split("_")[-1] for x in gene_expression.index])

{'0', '1', '2', '3', '4', '5', '6', '7', '8'}

In [19]:
# create a specimenID to individualID map
specimenID_to_individualID = dict(zip(rnaseq_met.specimenID, rnaseq_met.individualID))

In [20]:
# fix gene expression specimenIDs
gene_expression.index = list(map( lambda x: "_".join(x.split("_")[:-1]), gene_expression.index))
gene_expression.head()

gene_id,ENSG00000167578.11,ENSG00000242268.1,ENSG00000078237.4,ENSG00000263642.1,ENSG00000225275.4,ENSG00000060642.6,ENSG00000201788.1,ENSG00000263089.1,ENSG00000172137.14,ENSG00000240423.1,...,ENSG00000232668.1,ENSG00000089177.13,ENSG00000216352.1,ENSG00000267117.1,ENSG00000148943.7,ENSG00000265520.1,ENSG00000231119.2,ENSG00000105063.14,ENSG00000123685.4,ENSG00000181518.2
525_120515,60.84,0.08,4.39,0.0,0.0,5.98,0.0,0.0,19.69,0.09,...,0.0,4.98,0.0,0.0,10.02,0.0,0.20,40.59,4.46,0.00
383_120503,65.45,0.05,4.49,0.0,0.0,4.66,0.0,0.0,31.08,0.18,...,0.0,5.33,0.0,0.0,8.76,0.0,0.20,39.26,4.51,0.00
93_120417,69.18,0.08,2.51,0.0,0.0,2.77,0.0,0.0,10.76,0.12,...,0.0,6.95,0.0,0.0,6.28,0.0,0.24,34.70,5.27,0.05
610_120523,51.60,0.08,2.90,0.0,0.0,4.50,0.0,0.0,9.14,0.18,...,0.0,5.31,0.0,0.0,8.32,0.0,0.21,36.74,5.11,0.05
560_120517,48.61,0.10,2.67,0.0,0.0,4.26,0.0,0.0,5.87,0.17,...,0.0,7.12,0.0,0.0,8.68,0.0,0.09,39.85,3.77,0.00


In [21]:
# print gene expression shape
gene_expression.shape

(640, 55889)

In [22]:
# check how many gene expression specimenIDs can be mapped to individualIDs
mappable_specimenIDs = [specimenID for specimenID in gene_expression.index if specimenID in specimenID_to_individualID]
len(mappable_specimenIDs)

640

In [23]:
# map gene expression data specimenIDs to individualIDs
gene_expression = gene_expression.loc[mappable_specimenIDs]
gene_expression.index = list(map( lambda x: specimenID_to_individualID[x], gene_expression.index))
gene_expression.columns.name = None
gene_expression.index.name = "Sample ID"
gene_expression.head()

,ENSG00000167578.11,ENSG00000242268.1,ENSG00000078237.4,ENSG00000263642.1,ENSG00000225275.4,ENSG00000060642.6,ENSG00000201788.1,ENSG00000263089.1,ENSG00000172137.14,ENSG00000240423.1,...,ENSG00000232668.1,ENSG00000089177.13,ENSG00000216352.1,ENSG00000267117.1,ENSG00000148943.7,ENSG00000265520.1,ENSG00000231119.2,ENSG00000105063.14,ENSG00000123685.4,ENSG00000181518.2
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1743384,60.84,0.08,4.39,0.0,0.0,5.98,0.0,0.0,19.69,0.09,...,0.0,4.98,0.0,0.0,10.02,0.0,0.20,40.59,4.46,0.00
R6862468,65.45,0.05,4.49,0.0,0.0,4.66,0.0,0.0,31.08,0.18,...,0.0,5.33,0.0,0.0,8.76,0.0,0.20,39.26,4.51,0.00
R5415701,69.18,0.08,2.51,0.0,0.0,2.77,0.0,0.0,10.76,0.12,...,0.0,6.95,0.0,0.0,6.28,0.0,0.24,34.70,5.27,0.05
R1407047,51.60,0.08,2.90,0.0,0.0,4.50,0.0,0.0,9.14,0.18,...,0.0,5.31,0.0,0.0,8.32,0.0,0.21,36.74,5.11,0.05
R2197944,48.61,0.10,2.67,0.0,0.0,4.26,0.0,0.0,5.87,0.17,...,0.0,7.12,0.0,0.0,8.68,0.0,0.09,39.85,3.77,0.00


In [24]:
# check duplicates
gene_expression.duplicated().sum()

6

In [25]:
# drop duplicates
gene_expression = gene_expression.drop_duplicates()
gene_expression.duplicated().sum()

0

In [26]:
# check if some individualIDs are nans
gene_expression.index.isna().sum()

2

In [27]:
# remove observations where the the individual ID is missing
gene_expression = gene_expression[gene_expression.index.notna()]
gene_expression.shape

(638, 55889)

In [28]:
# filter clinical dtaa by mappable ids
gene_clinical_data = clinical_data.copy()
gene_clinical_data.index.name = "Sample ID"
common_ids = list(set(gene_expression.index).intersection(set(gene_clinical_data.index)))
gene_clinical_data = gene_clinical_data.loc[common_ids]
gene_expression = gene_expression.loc[common_ids]
gene_clinical_data.head()

,Gender,Race,PMI,Braak,Diagnosis
Sample ID,,,,,
R3426726,Female,1,1.583333,4,MCI+
R5256488,Female,1,9.083333,4,MCI
R9101940,Female,1,2.916667,1,Control
R5415701,Female,1,11.250000,0,MCI
R3328867,Male,1,8.000000,3,Control


In [29]:
# Merge gene expression with clinical data
gene_expression_data = pd.merge(gene_clinical_data.reset_index(), gene_expression.reset_index())
gene_expression_data.head()

,Sample ID,Gender,Race,PMI,Braak,Diagnosis,ENSG00000167578.11,ENSG00000242268.1,ENSG00000078237.4,ENSG00000263642.1,...,ENSG00000232668.1,ENSG00000089177.13,ENSG00000216352.1,ENSG00000267117.1,ENSG00000148943.7,ENSG00000265520.1,ENSG00000231119.2,ENSG00000105063.14,ENSG00000123685.4,ENSG00000181518.2
0,R3426726,Female,1,1.583333,4,MCI+,66.01,0.00,3.22,0.0,...,0.0,5.83,0.0,0.0,8.83,0.0,0.08,47.93,6.55,0.00
1,R5256488,Female,1,9.083333,4,MCI,60.73,0.17,1.84,0.0,...,0.0,6.88,0.0,0.0,6.39,0.0,0.25,26.90,3.96,0.00
2,R9101940,Female,1,2.916667,1,Control,61.29,0.00,4.42,0.0,...,0.0,4.84,0.0,0.0,9.37,0.0,0.20,37.38,3.84,0.15
3,R5415701,Female,1,11.250000,0,MCI,69.18,0.08,2.51,0.0,...,0.0,6.95,0.0,0.0,6.28,0.0,0.24,34.70,5.27,0.05
4,R3328867,Male,1,8.000000,3,Control,60.37,0.00,2.82,0.0,...,0.0,5.37,0.0,0.0,9.89,0.0,0.37,38.40,7.34,0.00


In [30]:
# check data dimensions
gene_expression_data.shape

(527, 55895)

In [31]:
# check duplicates
gene_expression_data.duplicated().sum()

0

In [32]:
# drop duplicates
gene_expression_data = gene_expression_data.drop_duplicates()
gene_expression_data.duplicated().sum()

0

In [33]:
# check if some individualIDs are nans
gene_expression_data.index.isna().sum()

0

In [34]:
# check data dimensions
gene_expression_data.shape

(527, 55895)

In [35]:
# fix duplicated values
numeric_columns = [feature for feature in gene_expression_data.columns if gene_expression_data[feature].dtype != 'O']
categorical_columns = [feature for feature in gene_expression_data.columns if gene_expression_data[feature].dtype == 'O']

# Group by 'Sample ID'
gene_expression_data = gene_expression_data.groupby('Sample ID').agg({
    **{col: lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0] for col in categorical_columns},  # Mode or first entry for categorical columns
    **{col: 'mean' for col in numeric_columns}       # Mean for numeric columns
})

gene_expression_data.shape

(525, 55895)

In [36]:
gene_expression_data.head()

,Sample ID,Gender,Diagnosis,Race,PMI,Braak,ENSG00000167578.11,ENSG00000242268.1,ENSG00000078237.4,ENSG00000263642.1,...,ENSG00000232668.1,ENSG00000089177.13,ENSG00000216352.1,ENSG00000267117.1,ENSG00000148943.7,ENSG00000265520.1,ENSG00000231119.2,ENSG00000105063.14,ENSG00000123685.4,ENSG00000181518.2
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1015854,R1015854,Female,MCI,1.0,4.633333,2.0,68.84,0.15,4.64,0.0,...,0.0,4.51,0.0,0.0,8.39,0.0,0.22,35.86,5.91,0.05
R1020037,R1020037,Female,MCI,1.0,2.750000,3.0,74.67,0.06,2.64,0.0,...,0.0,5.71,0.0,0.0,8.22,0.0,0.11,37.62,4.93,0.00
R1022980,R1022980,Female,AD,1.0,8.083333,3.0,38.84,0.11,2.98,0.0,...,0.0,4.38,0.0,0.0,7.31,0.0,0.08,23.13,2.42,0.00
R1028639,R1028639,Male,AD,1.0,6.916667,4.0,58.82,0.08,2.89,0.0,...,0.0,3.46,0.0,0.0,5.93,0.0,0.15,35.61,5.60,0.05
R1039781,R1039781,Female,AD,1.0,3.000000,5.0,70.30,0.00,2.69,0.0,...,0.0,7.19,0.0,0.0,7.01,0.0,0.30,42.40,4.50,0.00


In [37]:
ordered_columns = ["Sample ID", "Gender", "Race", "PMI", "Braak", "Diagnosis"] + list(gene_expression_data.columns[6:])
gene_expression_data = gene_expression_data[ordered_columns]
gene_expression_data.head()

,Sample ID,Gender,Race,PMI,Braak,Diagnosis,ENSG00000167578.11,ENSG00000242268.1,ENSG00000078237.4,ENSG00000263642.1,...,ENSG00000232668.1,ENSG00000089177.13,ENSG00000216352.1,ENSG00000267117.1,ENSG00000148943.7,ENSG00000265520.1,ENSG00000231119.2,ENSG00000105063.14,ENSG00000123685.4,ENSG00000181518.2
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1015854,R1015854,Female,1.0,4.633333,2.0,MCI,68.84,0.15,4.64,0.0,...,0.0,4.51,0.0,0.0,8.39,0.0,0.22,35.86,5.91,0.05
R1020037,R1020037,Female,1.0,2.750000,3.0,MCI,74.67,0.06,2.64,0.0,...,0.0,5.71,0.0,0.0,8.22,0.0,0.11,37.62,4.93,0.00
R1022980,R1022980,Female,1.0,8.083333,3.0,AD,38.84,0.11,2.98,0.0,...,0.0,4.38,0.0,0.0,7.31,0.0,0.08,23.13,2.42,0.00
R1028639,R1028639,Male,1.0,6.916667,4.0,AD,58.82,0.08,2.89,0.0,...,0.0,3.46,0.0,0.0,5.93,0.0,0.15,35.61,5.60,0.05
R1039781,R1039781,Female,1.0,3.000000,5.0,AD,70.30,0.00,2.69,0.0,...,0.0,7.19,0.0,0.0,7.01,0.0,0.30,42.40,4.50,0.00


In [38]:
def translate_to_uniprot(
    data,
    clinical_columns=["Sample ID", "Gender", "Race", "PMI", "Braak", "Diagnosis"],
    mapping_path="../reference_maps/ensembl_to_uniprot_1to1.json",
):
    """
    Translate Ensembl gene columns to UniProt IDs while ensuring clinical
    columns remain first.

    Parameters
    ----------
    data : pandas.DataFrame
        Input dataframe with Ensembl gene IDs as columns.
    clinical_columns : list
        Columns that should not be translated.
    mapping_path : str
        Path to Ensembl -> UniProt JSON mapping.

    Returns
    -------
    pandas.DataFrame
        DataFrame with UniProt columns replacing Ensembl columns and
        clinical columns appearing first.
    """

    import re
    import pandas as pd

    data = data.copy()
    mapper = load_json(mapping_path)

    # Remove Ensembl version numbers if present
    def strip_version(x):
        return re.sub(r"\.\d+$", "", x)

    # Identify gene columns
    gene_columns = [c for c in data.columns if c not in clinical_columns]

    mappable_columns = []
    rename_map = {}

    for col in gene_columns:
        clean = strip_version(col)
        if clean in mapper:
            mappable_columns.append(col)
            rename_map[col] = mapper[clean]

    # Keep only needed columns
    data = data[clinical_columns + mappable_columns]

    # Rename Ensembl -> UniProt
    data = data.rename(columns=rename_map)

    # Collapse duplicate UniProt columns if they appear
    data = data.groupby(axis=1, level=0).sum()

    # Ensure clinical columns remain first
    gene_cols = [c for c in data.columns if c not in clinical_columns]
    data = data[clinical_columns + gene_cols]

    return data

In [39]:
gene_expression_data = translate_to_uniprot(gene_expression_data)
gene_expression_data.drop(columns = ["Sample ID"], inplace = True)
gene_expression_data.head()

,Gender,Race,PMI,Braak,Diagnosis,A0A023HHK9,A0A023I889,A0A023I8P3,A0A023IN41,A0A023T695,...,Q9Y3D0,Q9Y3E1,Q9Y3E7,Q9Y4Z0,Q9Y5C5,Q9Y5I7,Q9Y5U9,Q9Y623,Q9Y6X8,U3KPV4
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1015854,Female,1.0,4.633333,2.0,MCI,0.19,31526.62,1126.37,26.44,147.51,...,100.17,31.72,0.17,63.50,3.24,0.19,14.38,0.02,5.60,0.00
R1020037,Female,1.0,2.750000,3.0,MCI,0.13,29613.34,1390.61,26.65,120.71,...,109.76,34.00,0.31,75.61,2.36,0.23,15.12,0.01,2.61,0.10
R1022980,Female,1.0,8.083333,3.0,AD,0.11,63943.76,1903.11,15.24,142.44,...,72.42,20.60,0.14,37.62,2.06,0.04,6.56,0.00,9.07,0.10
R1028639,Male,1.0,6.916667,4.0,AD,0.14,48540.78,2372.27,28.16,113.80,...,92.00,27.66,0.37,66.45,3.03,0.10,10.17,0.00,4.11,0.06
R1039781,Female,1.0,3.000000,5.0,AD,0.11,47942.73,1199.95,28.36,103.39,...,98.63,26.10,0.38,59.98,3.40,0.00,10.10,0.02,5.27,0.00


#### 3. Preprocess Transcript Expression Data

In [40]:
# Load transcript expression data
transcript_expression = pd.read_csv("../downloads/ROSMAP/ROSMAP_RNAseq_FPKM_isoform.tsv", sep = "\t")
rnaseq_sample_names = transcript_expression.columns[2:]
transcript_expression = transcript_expression.iloc[:,1:]
transcript_expression.index = transcript_expression.gene_id
transcript_expression = transcript_expression.drop("gene_id", axis=1).T
transcript_expression.head()

gene_id,ENSG00000175029.11,ENSG00000006744.11,ENSG00000134759.9,ENSG00000166912.12,ENSG00000040633.8,ENSG00000258690.1,ENSG00000008226.15,ENSG00000169696.10,ENSG00000178082.5,ENSG00000010818.4,...,ENSG00000239265.1,ENSG00000180172.5,ENSG00000220132.1,ENSG00000165862.9,ENSG00000101200.5,ENSG00000134207.8,ENSG00000160613.8,ENSG00000129194.3,ENSG00000187922.9,ENSG00000162714.7
525_120515_0,0.0,0.13,0.0,0.0,0.00,0.0,0.02,1.82,0.00,8.60,...,0.0,0.0,0.0,0.0,0.27,0.07,3.21,0.0,0.38,0.00
383_120503_0,0.0,0.00,0.0,0.0,0.00,0.0,0.10,1.59,0.18,9.08,...,0.0,0.0,0.0,0.0,0.29,0.22,2.76,0.0,0.45,0.00
93_120417_0,0.0,0.00,0.0,0.0,0.00,0.0,0.08,1.61,0.08,6.03,...,0.0,0.0,0.0,0.0,0.19,0.48,4.47,0.0,0.50,0.00
610_120523_0,0.0,0.00,0.0,0.0,0.05,0.0,0.05,1.87,0.10,5.37,...,0.0,0.0,0.0,0.0,0.67,0.44,3.33,0.0,0.15,0.04
560_120517_0,0.0,0.00,0.0,0.0,0.00,0.0,0.08,1.80,0.09,4.16,...,0.0,0.0,0.0,0.0,0.24,0.43,4.00,0.0,0.00,0.00


In [41]:
# Sample ID names extensions
set([x.split("_")[-1] for x in transcript_expression.index])

{'0', '1', '2', '3', '4', '5', '6', '7', '8'}

In [42]:
# fix transcript expression specimenIDs
transcript_expression.index = list(map( lambda x: "_".join(x.split("_")[:-1]), transcript_expression.index))
transcript_expression.head()

gene_id,ENSG00000175029.11,ENSG00000006744.11,ENSG00000134759.9,ENSG00000166912.12,ENSG00000040633.8,ENSG00000258690.1,ENSG00000008226.15,ENSG00000169696.10,ENSG00000178082.5,ENSG00000010818.4,...,ENSG00000239265.1,ENSG00000180172.5,ENSG00000220132.1,ENSG00000165862.9,ENSG00000101200.5,ENSG00000134207.8,ENSG00000160613.8,ENSG00000129194.3,ENSG00000187922.9,ENSG00000162714.7
525_120515,0.0,0.13,0.0,0.0,0.00,0.0,0.02,1.82,0.00,8.60,...,0.0,0.0,0.0,0.0,0.27,0.07,3.21,0.0,0.38,0.00
383_120503,0.0,0.00,0.0,0.0,0.00,0.0,0.10,1.59,0.18,9.08,...,0.0,0.0,0.0,0.0,0.29,0.22,2.76,0.0,0.45,0.00
93_120417,0.0,0.00,0.0,0.0,0.00,0.0,0.08,1.61,0.08,6.03,...,0.0,0.0,0.0,0.0,0.19,0.48,4.47,0.0,0.50,0.00
610_120523,0.0,0.00,0.0,0.0,0.05,0.0,0.05,1.87,0.10,5.37,...,0.0,0.0,0.0,0.0,0.67,0.44,3.33,0.0,0.15,0.04
560_120517,0.0,0.00,0.0,0.0,0.00,0.0,0.08,1.80,0.09,4.16,...,0.0,0.0,0.0,0.0,0.24,0.43,4.00,0.0,0.00,0.00


In [43]:
# print transcript expression shape
transcript_expression.shape

(640, 190051)

In [44]:
# check how many transcript expression specimenIDs can be mapped to individualIDs
mappable_specimenIDs = [specimenID for specimenID in transcript_expression.index if specimenID in specimenID_to_individualID]
len(mappable_specimenIDs)

640

In [45]:
# map transcript expression data specimenIDs to individualIDs
transcript_expression = transcript_expression.loc[mappable_specimenIDs]
transcript_expression.index = list(map( lambda x: specimenID_to_individualID[x], transcript_expression.index))
transcript_expression.columns.name = None
transcript_expression.index.name = "Sample ID"
transcript_expression.head()

,ENSG00000175029.11,ENSG00000006744.11,ENSG00000134759.9,ENSG00000166912.12,ENSG00000040633.8,ENSG00000258690.1,ENSG00000008226.15,ENSG00000169696.10,ENSG00000178082.5,ENSG00000010818.4,...,ENSG00000239265.1,ENSG00000180172.5,ENSG00000220132.1,ENSG00000165862.9,ENSG00000101200.5,ENSG00000134207.8,ENSG00000160613.8,ENSG00000129194.3,ENSG00000187922.9,ENSG00000162714.7
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1743384,0.0,0.13,0.0,0.0,0.00,0.0,0.02,1.82,0.00,8.60,...,0.0,0.0,0.0,0.0,0.27,0.07,3.21,0.0,0.38,0.00
R6862468,0.0,0.00,0.0,0.0,0.00,0.0,0.10,1.59,0.18,9.08,...,0.0,0.0,0.0,0.0,0.29,0.22,2.76,0.0,0.45,0.00
R5415701,0.0,0.00,0.0,0.0,0.00,0.0,0.08,1.61,0.08,6.03,...,0.0,0.0,0.0,0.0,0.19,0.48,4.47,0.0,0.50,0.00
R1407047,0.0,0.00,0.0,0.0,0.05,0.0,0.05,1.87,0.10,5.37,...,0.0,0.0,0.0,0.0,0.67,0.44,3.33,0.0,0.15,0.04
R2197944,0.0,0.00,0.0,0.0,0.00,0.0,0.08,1.80,0.09,4.16,...,0.0,0.0,0.0,0.0,0.24,0.43,4.00,0.0,0.00,0.00


In [46]:
# check duplicates
transcript_expression.duplicated().sum()

6

In [47]:
# drop duplicates
transcript_expression = transcript_expression.drop_duplicates()
transcript_expression.duplicated().sum()

0

In [48]:
# check if some individualIDs are nans
transcript_expression.index.isna().sum()

2

In [49]:
# remove observations where the the individual ID is missing
transcript_expression = transcript_expression[transcript_expression.index.notna()]
transcript_expression.shape

(638, 190051)

In [50]:
# filter clinical dtaa by mappable ids
transcript_clinical_data = clinical_data.copy()
transcript_clinical_data.index.name = "Sample ID"
common_ids = list(set(transcript_expression.index).intersection(set(transcript_clinical_data.index)))
transcript_clinical_data = transcript_clinical_data.loc[common_ids]
transcript_expression = transcript_expression.loc[common_ids]
transcript_clinical_data.head()

,Gender,Race,PMI,Braak,Diagnosis
Sample ID,,,,,
R3426726,Female,1,1.583333,4,MCI+
R5256488,Female,1,9.083333,4,MCI
R9101940,Female,1,2.916667,1,Control
R5415701,Female,1,11.250000,0,MCI
R3328867,Male,1,8.000000,3,Control


In [51]:
# Merge transcript expression with clinical data
transcript_expression_data = pd.merge(transcript_clinical_data.reset_index(), transcript_expression.reset_index())
transcript_expression_data.head()

,Sample ID,Gender,Race,PMI,Braak,Diagnosis,ENSG00000175029.11,ENSG00000006744.11,ENSG00000134759.9,ENSG00000166912.12,...,ENSG00000239265.1,ENSG00000180172.5,ENSG00000220132.1,ENSG00000165862.9,ENSG00000101200.5,ENSG00000134207.8,ENSG00000160613.8,ENSG00000129194.3,ENSG00000187922.9,ENSG00000162714.7
0,R3426726,Female,1,1.583333,4,MCI+,0.0,0.0,0.0,0.0,...,0.00,0.0,0.0,0.0,0.46,0.42,3.36,0.13,0.32,0.00
1,R5256488,Female,1,9.083333,4,MCI,0.0,0.0,0.0,0.0,...,0.03,0.0,0.0,0.0,0.11,0.51,3.91,0.00,0.12,0.00
2,R9101940,Female,1,2.916667,1,Control,0.0,0.0,0.0,0.0,...,0.06,0.0,0.0,0.0,0.20,0.20,3.60,0.15,0.36,0.06
3,R5415701,Female,1,11.250000,0,MCI,0.0,0.0,0.0,0.0,...,0.00,0.0,0.0,0.0,0.19,0.48,4.47,0.00,0.50,0.00
4,R3328867,Male,1,8.000000,3,Control,0.0,0.0,0.0,0.0,...,0.00,0.0,0.0,0.0,0.20,0.56,3.45,0.00,0.63,0.08


In [52]:
# check data dimensions
transcript_expression_data.shape

(527, 190057)

In [53]:
# check duplicates
transcript_expression_data.duplicated().sum()

0

In [54]:
# drop duplicates
transcript_expression_data = transcript_expression_data.drop_duplicates()
transcript_expression_data.duplicated().sum()

0

In [55]:
# check if some individualIDs are nans
transcript_expression_data.index.isna().sum()

0

In [56]:
# check data dimensions
transcript_expression_data.shape

(527, 190057)

In [57]:
# keep sample/clinical columns aside
clinical_cols = ["Sample ID", "Gender", "Race", "PMI", "Braak", "Diagnosis"]

clinical = transcript_expression_data[clinical_cols]
features = transcript_expression_data.drop(columns=clinical_cols)

# collapse duplicate feature columns by sum (or mean)
features = features.groupby(axis=1, level=0).sum()

# rebuild
transcript_expression_data = pd.concat([clinical, features], axis=1)

In [58]:
# fix duplicated values
numeric_columns = transcript_expression_data.select_dtypes(include="number").columns.tolist()
categorical_columns = transcript_expression_data.select_dtypes(exclude="number").columns.tolist()

# Group by 'Sample ID'
transcript_expression_data = transcript_expression_data.groupby('Sample ID').agg({
    **{col: lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0] for col in categorical_columns},  # Mode or first entry for categorical columns
    **{col: 'mean' for col in numeric_columns}       # Mean for numeric columns
})

transcript_expression_data.shape

(525, 55895)

In [59]:
transcript_expression_data.head()

,Sample ID,Gender,Diagnosis,Race,PMI,Braak,ENSG00000000003.10,ENSG00000000005.5,ENSG00000000419.8,ENSG00000000457.8,...,ENSGR0000237801.1,ENSGR0000248421.1,ENSGR0000249358.1,ENSGR0000263835.1,ENSGR0000263980.1,ENSGR0000264510.1,ENSGR0000264819.1,ENSGR0000265350.1,ENSGR0000265658.1,ENSGR0000266731.1
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1015854,R1015854,Female,MCI,1.0,4.633333,2.0,4.18,0.15,15.82,1.61,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R1020037,R1020037,Female,MCI,1.0,2.750000,3.0,3.00,0.03,18.45,1.91,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R1022980,R1022980,Female,AD,1.0,8.083333,3.0,5.86,0.17,7.98,1.26,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R1028639,R1028639,Male,AD,1.0,6.916667,4.0,3.19,0.17,13.12,1.16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R1039781,R1039781,Female,AD,1.0,3.000000,5.0,3.43,0.14,9.11,1.19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
ordered_columns = ["Sample ID", "Gender", "Race", "PMI", "Braak", "Diagnosis"] + list(transcript_expression_data.columns[6:])
transcript_expression_data = transcript_expression_data[ordered_columns]
transcript_expression_data.head()

,Sample ID,Gender,Race,PMI,Braak,Diagnosis,ENSG00000000005.5,ENSG00000000419.8,ENSG00000000457.8,ENSG00000000460.11,...,ENSGR0000248421.1,ENSGR0000249358.1,ENSGR0000263835.1,ENSGR0000263980.1,ENSGR0000264510.1,ENSGR0000264819.1,ENSGR0000265350.1,ENSGR0000265658.1,ENSGR0000266731.1,Sample ID
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1015854,R1015854,Female,1.0,4.633333,2.0,MCI,0.15,15.82,1.61,1.18,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,R1015854
R1020037,R1020037,Female,1.0,2.750000,3.0,MCI,0.03,18.45,1.91,0.69,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,R1020037
R1022980,R1022980,Female,1.0,8.083333,3.0,AD,0.17,7.98,1.26,1.15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,R1022980
R1028639,R1028639,Male,1.0,6.916667,4.0,AD,0.17,13.12,1.16,0.35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,R1028639
R1039781,R1039781,Female,1.0,3.000000,5.0,AD,0.14,9.11,1.19,0.72,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,R1039781


In [65]:
transcript_expression_data = translate_to_uniprot(transcript_expression_data)
transcript_expression_data.drop(columns = ["Sample ID"], inplace = True)
transcript_expression_data.head()

,Gender,Race,PMI,Braak,Diagnosis,A0A023HHK9,A0A023I889,A0A023I8P3,A0A023IN41,A0A023T695,...,Q9Y3D0,Q9Y3E1,Q9Y3E7,Q9Y4Z0,Q9Y5C5,Q9Y5I7,Q9Y5U9,Q9Y623,Q9Y6X8,U3KPV4
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1015854,Female,1.0,4.633333,2.0,MCI,0.19,31526.62,1126.37,26.44,147.53,...,100.17,31.72,0.17,63.50,3.24,0.19,14.38,0.02,5.60,0.00
R1020037,Female,1.0,2.750000,3.0,MCI,0.13,29613.34,1390.61,26.64,120.70,...,109.76,34.00,0.31,75.61,2.36,0.23,15.12,0.01,2.61,0.10
R1022980,Female,1.0,8.083333,3.0,AD,0.11,63943.76,1903.11,15.24,142.44,...,72.43,20.60,0.14,37.62,2.06,0.04,6.56,0.00,9.07,0.10
R1028639,Male,1.0,6.916667,4.0,AD,0.14,48540.78,2372.27,28.18,113.81,...,92.00,27.67,0.37,66.45,3.03,0.10,10.17,0.00,4.11,0.06
R1039781,Female,1.0,3.000000,5.0,AD,0.11,47942.73,1199.95,28.36,103.39,...,98.63,26.10,0.38,59.98,3.40,0.00,10.10,0.02,5.27,0.00


#### 4. Preprocess Metabolomics Data

In [66]:
# Function to Impute Missing Values
def Imputer(data):
    
    imputer = KNNImputer(n_neighbors=5)
    
    return imputer.fit_transform(data)

In [67]:
Metabolon_path = "../downloads/ROSMAP/ROSMAP_Metabolon_HD4_Brain514_assay_data.csv"
Metabolon_expression_data = pd.read_csv(Metabolon_path)
Metabolon_expression_data.head()

,individualID,PARENT_SAMPLE_NAME,BATCH_209,BATCH_305,BATCH_400,BATCH_402,SAMPLE_TYPE,MATRIX_DESIGNATION,35,41,...,999925828,999925855,999925884,999925936,999925948,999925957,999925981,999925982,999925983,999926062
0,R1687970,DUKE-07620,161918,161922,162001,161910,EXPERIMENTAL,NaN,0.612634,0.960771,...,1.177809,1.482806,0.915167,0.828067,1.099776,NaN,0.990705,0.915459,NaN,NaN
1,R7738727,DUKE-07621,161918,161922,162001,161910,EXPERIMENTAL,NaN,NaN,3.966244,...,0.869506,1.185795,1.479626,1.436135,0.454324,NaN,1.048476,0.739520,1.777648,NaN
2,R2575548,DUKE-07622,161918,161922,162001,161910,EXPERIMENTAL,NaN,1.335772,1.582573,...,1.395358,1.445302,1.151046,0.669163,0.638373,NaN,0.827218,0.753234,1.015258,0.480840
3,R5061712,DUKE-07623,161918,161922,162001,161910,EXPERIMENTAL,NaN,1.190039,2.253279,...,1.151727,0.840098,1.163917,0.639246,0.885062,0.221254,1.312609,0.883237,NaN,1.013376
4,R9991313,DUKE-07624,161918,161922,162001,161910,EXPERIMENTAL,NaN,1.331312,0.475191,...,1.359079,0.535779,0.697159,0.598017,1.512382,4.935088,0.921047,0.596167,NaN,NaN


In [68]:
# the expression data begins from column index 8
Metabolon_expression_data.index = Metabolon_expression_data.individualID
Metabolon_expression_data.index.name = "individualID"
Metabolon_expression_data = Metabolon_expression_data.iloc[:,8:]
Metabolon_expression_data.head()

,35,41,49,50,55,62,73,93,98,111,...,999925828,999925855,999925884,999925936,999925948,999925957,999925981,999925982,999925983,999926062
individualID,,,,,,,,,,,,,,,,,,,,,
R1687970,0.612634,0.960771,1.140274,0.674424,0.882915,NaN,NaN,0.656020,0.529580,1.807763,...,1.177809,1.482806,0.915167,0.828067,1.099776,NaN,0.990705,0.915459,NaN,NaN
R7738727,NaN,3.966244,5.938049,1.442034,1.806857,NaN,NaN,0.288551,1.093758,0.296401,...,0.869506,1.185795,1.479626,1.436135,0.454324,NaN,1.048476,0.739520,1.777648,NaN
R2575548,1.335772,1.582573,0.913341,1.159618,1.359713,NaN,NaN,0.284284,1.083162,0.173330,...,1.395358,1.445302,1.151046,0.669163,0.638373,NaN,0.827218,0.753234,1.015258,0.480840
R5061712,1.190039,2.253279,1.097244,0.903730,0.573183,NaN,0.737894,0.454682,0.861229,1.191450,...,1.151727,0.840098,1.163917,0.639246,0.885062,0.221254,1.312609,0.883237,NaN,1.013376
R9991313,1.331312,0.475191,0.864131,2.332162,0.831024,NaN,0.235693,0.274049,0.823301,0.383048,...,1.359079,0.535779,0.697159,0.598017,1.512382,4.935088,0.921047,0.596167,NaN,NaN


In [69]:
Metabolon_expression_data.shape

(521, 1055)

In [70]:
Metabolon_expression_data.isna().sum()

35            43
41             7
49            20
50             1
55            17
            ... 
999925957    359
999925981      7
999925982      2
999925983    174
999926062    235
Length: 1055, dtype: int64

In [71]:
# Apparantly some columns contain all nan values. We remove them
Metabolon_expression_data = Metabolon_expression_data.dropna(axis=1, how='all')
Metabolon_expression_data.isna().sum().sort_values().tail(20)

100001216    515
100001207    515
100020493    515
100001527    515
100019779    515
100020005    516
100004286    516
100015777    516
100009052    516
100021792    517
100008943    517
100021791    518
100021747    518
100002321    519
100001933    519
100015876    519
100020914    519
100002968    519
100002155    520
100015879    520
dtype: int64

**Observation** 
* There are a total of 521 samples. And several features with high missingness.

In [72]:
# compute missing rate (%)
Metabolon_expression_data.isna().sum().sort_values().tail(20)*(100/521)

100001216    98.848369
100001207    98.848369
100020493    98.848369
100001527    98.848369
100019779    98.848369
100020005    99.040307
100004286    99.040307
100015777    99.040307
100009052    99.040307
100021792    99.232246
100008943    99.232246
100021791    99.424184
100021747    99.424184
100002321    99.616123
100001933    99.616123
100015876    99.616123
100020914    99.616123
100002968    99.616123
100002155    99.808061
100015879    99.808061
dtype: float64

In [73]:
# set threshold to remove features
threshold = 20
# get feature with missing rate more than threshold
missingness = Metabolon_expression_data.isna().sum()*(100/521)
to_remove = [missingness.index[i] for i in range(len(missingness)) if missingness[i] >= threshold]
len(to_remove)

405

In [74]:
# remove features with many missing data
Metabolon_expression_data = Metabolon_expression_data.drop(to_remove, axis=1)
Metabolon_expression_data.isna().sum().sort_values().tail(10)

100006296     91
100000442     92
1096          94
1263          95
100003823     95
213           96
100001413     98
1383         100
100019800    101
1830         104
dtype: int64

In [75]:
# map metabolite names
metabolon_metadata = pd.read_csv("../downloads/ROSMAP/ROSMAP Metabolon HD4 Data Dictionary.csv")
metabolon_metadata.head()

,CHEM_ID,LIB_ID,COMP_ID,CHRO_LIB_ENTRY_ID,SUPER_PATHWAY,SUB_PATHWAY,PATHWAY_SORTORDER,PATHWAY_STATUS,TYPE,INCHIKEY,SMILES,CHEMICAL_NAME,SHORT_NAME,CAS,CHEMSPIDER,HMDB,KEGG,PLANT_CYC,PUBCHEM,PLATFORM
0,35,400,42370,166164,Amino Acid,Glutamate Metabolism,62.0,NaN,NAMED,DWAKNKKXGALPNW-REWHXWOFAV,OC(C1CCC=N1)=O,S-1-pyrroline-5-carboxylate,S-1-pyrroline-5-carboxylate,2906-39-0,10140206,HMDB0001301,C04322,NaN,11966181,LC/MS Pos Early
1,41,400,1633,155434,Amino Acid,Histidine Metabolism,85.0,NaN,NAMED,CCLQKVKJOGVQLU-QMMMGPOBBT,NCCCC(NC(CC1=CN=CN1)C(O)=O)=O,homocarnosine,homocarnosine,3650-73-5,8418848,HMDB0000745,C00884,NaN,"89235,10243361",LC/MS Pos Early
2,49,400,1408,155357,Amino Acid,Polyamine Metabolism,537.0,NaN,NAMED,KIDHWZJUCRJVML-UHFFFAOYAX,NCCCCN,putrescine,putrescine,110-60-1,13837702,HMDB0001414,C00134,PUTRESCINE,1045,LC/MS Pos Early
3,50,400,485,155305,Amino Acid,Polyamine Metabolism,545.0,NaN,NAMED,ATHGHQPFGPMSJY-UHFFFAOYAK,NCCCCNCCCN,spermidine,spermidine,124-20-9,1071,HMDB0001257,C00315,SPERMIDINE,1102,LC/MS Pos Early
4,55,400,27665,155829,Cofactors and Vitamins,Nicotinate and Nicotinamide Metabolism,4314.0,NaN,NAMED,LDHMAVIPBRSVRG-UHFFFAOYAE,C[N+]1=CC(C([NH-])=O)=CC=C1,1-methylnicotinamide,1-methylnicotinamide,1005-24-9,8305504,HMDB0000699,C02918,NaN,457,LC/MS Pos Early


In [76]:
# create metabolon map
metabolon_map = dict(zip(metabolon_metadata.CHEM_ID, metabolon_metadata.CHEMICAL_NAME))

# map names
Metabolon_expression_data.columns = list(map(lambda x: metabolon_map[int(x)], Metabolon_expression_data.columns))
Metabolon_expression_data.head()

,S-1-pyrroline-5-carboxylate,homocarnosine,putrescine,spermidine,1-methylnicotinamide,alpha-ketoglutarate,kynurenate,3-hydroxyisobutyrate,3-hydroxy-3-methylglutarate,3-phosphoglycerate,...,X - 25244,X - 25422,X - 25790,X - 25828,X - 25855,X - 25884,X - 25936,X - 25948,X - 25981,X - 25982
individualID,,,,,,,,,,,,,,,,,,,,,
R1687970,0.612634,0.960771,1.140274,0.674424,0.882915,0.656020,0.529580,1.807763,1.455239,1.294100,...,0.533627,0.382170,1.115068,1.177809,1.482806,0.915167,0.828067,1.099776,0.990705,0.915459
R7738727,NaN,3.966244,5.938049,1.442034,1.806857,0.288551,1.093758,0.296401,1.575174,0.685722,...,0.864869,0.526324,0.831334,0.869506,1.185795,1.479626,1.436135,0.454324,1.048476,0.739520
R2575548,1.335772,1.582573,0.913341,1.159618,1.359713,0.284284,1.083162,0.173330,1.131180,0.542446,...,1.354027,0.674569,2.005801,1.395358,1.445302,1.151046,0.669163,0.638373,0.827218,0.753234
R5061712,1.190039,2.253279,1.097244,0.903730,0.573183,0.454682,0.861229,1.191450,1.671078,1.013997,...,1.475159,0.890307,1.046844,1.151727,0.840098,1.163917,0.639246,0.885062,1.312609,0.883237
R9991313,1.331312,0.475191,0.864131,2.332162,0.831024,0.274049,0.823301,0.383048,0.742593,1.929360,...,4.119633,1.182044,0.493048,1.359079,0.535779,0.697159,0.598017,1.512382,0.921047,0.596167


In [77]:
# Impute Missing Values
Metabolon_expression_data = pd.DataFrame(Imputer(Metabolon_expression_data), columns = Metabolon_expression_data.columns, index = Metabolon_expression_data.index)
Metabolon_expression_data.isna().sum().sum()

0

In [78]:
# Now we will merge matching 
matched_individuals = [sample_id for sample_id in Metabolon_expression_data.index if sample_id in clinical_data.index]
Metabolon_data = pd.concat([clinical_data.loc[matched_individuals].iloc[:,1:], Metabolon_expression_data.loc[matched_individuals]], axis=1)
Metabolon_data.index.name = "Sample ID"
Metabolon_data.head()

,Race,PMI,Braak,Diagnosis,S-1-pyrroline-5-carboxylate,homocarnosine,putrescine,spermidine,1-methylnicotinamide,alpha-ketoglutarate,...,X - 25244,X - 25422,X - 25790,X - 25828,X - 25855,X - 25884,X - 25936,X - 25948,X - 25981,X - 25982
Sample ID,,,,,,,,,,,,,,,,,,,,,
R1687970,1,7.583333,3,MCI,0.612634,0.960771,1.140274,0.674424,0.882915,0.656020,...,0.533627,0.382170,1.115068,1.177809,1.482806,0.915167,0.828067,1.099776,0.990705,0.915459
R7738727,1,4.083333,3,MCI,0.813785,3.966244,5.938049,1.442034,1.806857,0.288551,...,0.864869,0.526324,0.831334,0.869506,1.185795,1.479626,1.436135,0.454324,1.048476,0.739520
R2575548,1,6.250000,4,Control,1.335772,1.582573,0.913341,1.159618,1.359713,0.284284,...,1.354027,0.674569,2.005801,1.395358,1.445302,1.151046,0.669163,0.638373,0.827218,0.753234
R5973191,1,8.583333,2,Control,1.155859,0.984403,0.678143,0.940866,0.877829,0.751438,...,1.985345,0.629327,1.268829,0.895727,1.322222,1.090604,1.363795,1.449092,1.555578,0.813836
R9771008,1,3.083333,5,AD,0.852482,0.769457,0.620523,1.486285,0.792433,0.575967,...,3.292233,3.811044,1.619271,0.772019,1.562373,1.193642,1.638651,0.651751,1.100757,1.511485


#### 5. Preprocess Proteomics Data

In [79]:
# Load protein metadata
protein_metadata = pd.read_csv("../downloads/ROSMAP/OhNM2025_ROSMAP_plasma_Soma7k_protein_metadata.csv") 
protein_metadata.head()

,SeqId,SeqIdVersion,SomaId,TargetFullName,Target,UniProt,EntrezGeneID,EntrezGeneSymbol,Organism,Units,...,Cal_PLT22095,CalQcRatio_PLT22106_200170,Cal_PLT22094,CalQcRatio_PLT22105_200170,Cal_PLT22183,CalQcRatio_PLT22095_200170,ColCheck,CalQcRatio_PLT22094_200170,CalQcRatio_PLT22183_200170,QcReference_200170
0,10000-28,3,SL019233,Beta-crystallin B2,CRBB2,P43320,1415,CRYBB2,Human,RFU,...,1.001488,1.012,1.047152,1.021,1.060092,1.043,PASS,0.991,1.026,2304.3
1,10001-7,3,SL002564,RAF proto-oncogene serine/threonine-protein ki...,c-Raf,P04049,5894,RAF1,Human,RFU,...,0.938539,1.072,0.908780,1.060,0.884825,1.040,PASS,1.078,1.005,327.4
2,10003-15,3,SL019245,Zinc finger protein 41,ZNF41,P51814,7592,ZNF41,Human,RFU,...,0.951456,0.921,0.975355,0.990,0.866162,0.954,PASS,0.946,0.891,173.6
3,10006-25,3,SL019228,ETS domain-containing protein Elk-1,ELK1,P19419,2002,ELK1,Human,RFU,...,1.013871,0.998,1.027476,1.056,1.053735,1.053,PASS,1.051,1.056,543.1
4,10008-43,3,SL019234,Guanylyl cyclase-activating protein 1,GUC1A,P43080,2978,GUCA1A,Human,RFU,...,0.965670,1.022,0.971107,1.037,0.983789,1.020,PASS,1.033,1.018,473.6


In [80]:
# Select SeqID, UniprotID and GeneSymbol
protein_metadata = protein_metadata[["SeqId", "UniProt", "EntrezGeneSymbol"]]
protein_metadata.head()

,SeqId,UniProt,EntrezGeneSymbol
0,10000-28,P43320,CRYBB2
1,10001-7,P04049,RAF1
2,10003-15,P51814,ZNF41
3,10006-25,P19419,ELK1
4,10008-43,P43080,GUCA1A


In [81]:
# create maps to Uniprot and Symbols
protein_id_maps = {
    "seq_id_to_uniprot": dict(zip(protein_metadata.SeqId, protein_metadata.UniProt)),
    "seq_id_to_GeneSymbol": dict(zip(protein_metadata.dropna().SeqId, protein_metadata.dropna().EntrezGeneSymbol))
} 

# save mappings
save_json("../artifacts/rosmap_protein_id_maps.json", protein_id_maps)

In [82]:
# Load ANML normalized data
anml_normalized_data = pd.read_csv("../downloads/ROSMAP/OhNM2025_ROSMAP_plasma_Soma7k_protein_level_ANML_log10.csv")
anml_normalized_data.head()

,projid_visit,10000-28,10001-7,10003-15,10006-25,10008-43,10010-10,10011-65,10012-5,10014-31,...,9984-12,9986-14,9987-30,9989-12,9991-112,9993-11,9994-217,9995-6,9997-12,9999-1
0,R4014413_01,2.838219,2.552181,2.247973,2.778224,2.664172,2.563481,3.206043,3.142921,2.970440,...,2.798443,3.446755,2.718834,2.745777,2.649627,3.256958,3.191591,3.166756,3.867968,3.149219
1,R5357999_02,2.723784,3.127494,2.191171,2.740600,2.662569,2.531479,3.411081,3.132644,2.972804,...,2.770705,4.072033,2.705265,2.631139,2.561578,3.029546,3.145631,3.287085,4.088122,3.247531
2,R7139444_13,2.789933,2.722552,2.222196,2.736078,2.677881,2.517724,3.319023,3.203441,2.971971,...,2.867939,3.529097,2.741152,2.686279,2.690816,3.220553,3.230423,3.136752,3.852108,3.058768
3,R7095349_03,2.735759,2.502017,2.183555,2.716838,2.655523,2.553155,3.311775,3.099819,2.939170,...,2.736078,3.863858,2.665018,2.687529,2.590284,3.112069,3.152319,3.720821,3.932596,3.117172
4,R4364014_00,2.682326,2.510277,2.217221,2.723291,2.686815,2.513084,3.378271,3.135673,2.932981,...,2.823279,3.980044,2.701654,2.663041,2.573220,3.056180,3.149465,3.069742,3.801829,3.047898


In [83]:
# map projid_visit to projid
# load sample metadata
sample_metadata = pd.read_csv("../downloads/ROSMAP/OhNM2025_ROSMAP_plasma_Soma7k_sample_metadata.csv")
projid_visit_to_projid_mapper = dict(zip(sample_metadata.projid_visit, sample_metadata.projid))
sample_metadata.head()

,projid_visit,projid,Visit,study,msex,age_at_visit,Diagnosis,cogn_global,apoe_genotype,educ,age_death,Storage_days
0,R4014413_01,R4014413,1,MAP,0,87.876797,MCI,-1.054759,34.0,16,92.783025,6566.0
1,R5357999_02,R5357999,2,MAP,0,99.071869,AD+,-0.988238,23.0,16,102.091718,6566.0
2,R7139444_13,R7139444,13,MAP,0,97.935661,NCI,1.133552,33.0,21,107.679671,4222.0
3,R7095349_03,R7095349,3,ROS,0,94.182067,AD,NaN,33.0,16,95.310062,6535.0
4,R4364014_00,R4364014,0,MAP,0,92.605065,AD,-1.429396,33.0,12,97.456537,6535.0


In [84]:
# map protein columns
seq_ids = anml_normalized_data.columns[1:]
uniprot_ids = [protein_id_maps["seq_id_to_uniprot"][seq_id] for seq_id in seq_ids] 
anml_normalized_data.columns = ["projid_visit"] + uniprot_ids 
anml_normalized_data.projid_visit = anml_normalized_data.projid_visit.apply(lambda x: projid_visit_to_projid_mapper[x])
anml_normalized_data.head()

,projid_visit,P43320,P04049,P51814,P19419,P43080,Q14457,Q01968,O95238,O43623,...,Q96EC8,Q8N729,Q8N386,Q50LG9,Q9NT22,O43296,P51164,P33316,Q92575,O14896
0,R4014413,2.838219,2.552181,2.247973,2.778224,2.664172,2.563481,3.206043,3.142921,2.970440,...,2.798443,3.446755,2.718834,2.745777,2.649627,3.256958,3.191591,3.166756,3.867968,3.149219
1,R5357999,2.723784,3.127494,2.191171,2.740600,2.662569,2.531479,3.411081,3.132644,2.972804,...,2.770705,4.072033,2.705265,2.631139,2.561578,3.029546,3.145631,3.287085,4.088122,3.247531
2,R7139444,2.789933,2.722552,2.222196,2.736078,2.677881,2.517724,3.319023,3.203441,2.971971,...,2.867939,3.529097,2.741152,2.686279,2.690816,3.220553,3.230423,3.136752,3.852108,3.058768
3,R7095349,2.735759,2.502017,2.183555,2.716838,2.655523,2.553155,3.311775,3.099819,2.939170,...,2.736078,3.863858,2.665018,2.687529,2.590284,3.112069,3.152319,3.720821,3.932596,3.117172
4,R4364014,2.682326,2.510277,2.217221,2.723291,2.686815,2.513084,3.378271,3.135673,2.932981,...,2.823279,3.980044,2.701654,2.663041,2.573220,3.056180,3.149465,3.069742,3.801829,3.047898


In [85]:
proteomics_data = anml_normalized_data.copy()
proteomics_data.set_index("projid_visit", inplace = True)
common_ids = list(set(proteomics_data.index).intersection(set(clinical_data.index)))
proteomics_data = pd.concat([clinical_data.loc[common_ids], proteomics_data.loc[common_ids]], axis = 1)
proteomics_data.index.name = "Sample ID"
proteomics_data.head()

,Gender,Race,PMI,Braak,Diagnosis,P43320,P04049,P51814,P19419,P43080,...,Q96EC8,Q8N729,Q8N386,Q50LG9,Q9NT22,O43296,P51164,P33316,Q92575,O14896
Sample ID,,,,,,,,,,,,,,,,,,,,,
R5256488,Female,1,9.083333,4,MCI,2.666518,2.539703,2.175512,2.772688,2.675045,...,2.696356,3.709889,2.626443,2.637490,3.347837,2.973728,3.063896,3.462188,4.316339,3.261952
R7347788,Male,1,5.000000,1,Control,2.938670,2.479143,2.231724,2.716337,2.756788,...,2.853090,3.729869,2.767749,2.711217,2.799685,3.177883,3.173623,3.195014,3.729667,2.921738
R4790045,Male,1,3.500000,4,MCI,2.717754,2.406199,2.146748,2.774079,2.650113,...,2.735359,3.977431,2.848435,2.590619,2.742175,3.153937,3.135959,3.218614,4.040171,3.017451
R8781115,Male,1,5.333333,5,Control,2.666705,2.500099,2.253580,2.818754,2.663889,...,2.768860,3.938154,2.697752,2.662286,3.069409,2.979457,3.150664,3.129787,3.968558,3.110893
R3927700,Female,1,7.583333,3,Control,2.804344,2.512418,2.250420,2.700704,2.576572,...,2.764774,3.761010,2.726890,2.675137,2.685204,3.446133,3.592232,3.105067,3.879463,3.154302


#### 6. Preprocess Methylation Data

In [86]:
meth = pd.read_csv("../downloads/ROSMAP/ROSMAP_arrayMethylation_imputed.tsv.gz", sep="\t")
meth.head()

,TargetID,TBI-AUTO73325-PT-3149,PT-BZHL,PT-BZCH,PT-BY9H,TBI-AUTO73307-PT-314I,TBI-AUTO73043-PT-35BD,PT-BZI5,PT-BZ1A,PT-318X,...,TBI-AUTO72955-PT-35OC,TBI-AUTO73291-PT-35OD,PT-BZD7,PT-BZG8,PT-C1N8,PT-BYJP,PT-BZHV,PT-BZI2,TBI-AUTO73257-PT-35OE,PT-BZD3
0,cg00000165,0.231359,0.157857,0.127105,0.149988,0.130532,0.174451,0.170026,0.165900,0.157745,...,0.124635,0.168002,0.152200,0.230482,0.177287,0.195611,0.151338,0.151508,0.167960,0.136220
1,cg00000363,0.140664,0.114399,0.140580,0.145805,0.122833,0.117144,0.155559,0.131309,0.157749,...,0.129301,0.126910,0.135973,0.162883,0.122399,0.112684,0.125054,0.118550,0.114618,0.133814
2,cg00000957,0.776220,0.818279,0.630316,0.849261,0.861136,0.751543,0.778917,0.859892,0.787279,...,0.735520,0.740686,0.748494,0.797072,0.793081,0.692638,0.785597,0.712386,0.839168,0.888304
3,cg00001349,0.865488,0.919570,0.882256,0.902912,0.907610,0.897496,0.939212,0.919514,0.805626,...,0.885292,0.900684,0.906257,0.756173,0.948341,0.904996,0.927201,0.856637,0.917827,0.943320
4,cg00001364,0.772899,0.711099,0.694639,0.651986,0.748538,0.706625,0.743491,0.712513,0.722169,...,0.720420,0.724226,0.694462,0.776165,0.758569,0.733207,0.750224,0.737402,0.685959,0.770543


In [87]:
meth.set_index("TargetID", inplace = True)
meth = meth.T 
meth.head()

TargetID,cg00000165,cg00000363,cg00000957,cg00001349,cg00001364,cg00001446,cg00001534,cg00001583,cg00001593,cg00002028,...,ch.22.31817810F,ch.22.33863861F,ch.22.533187F,ch.22.740407F,ch.22.757911F,ch.22.772318F,ch.22.43177094F,ch.22.909671F,ch.22.46830341F,ch.22.1008279F
TBI-AUTO73325-PT-3149,0.231359,0.140664,0.776220,0.865488,0.772899,0.842025,0.883920,0.099888,0.918081,0.026821,...,0.161798,0.257386,0.128748,0.139123,0.089335,0.233366,0.173744,0.409177,0.225070,0.067878
PT-BZHL,0.157857,0.114399,0.818279,0.919570,0.711099,0.831457,0.877727,0.076240,0.955305,0.035839,...,0.198222,0.292629,0.125090,0.127840,0.089504,0.198370,0.199095,0.463476,0.279521,0.072976
PT-BZCH,0.127105,0.140580,0.630316,0.882256,0.694639,0.835075,0.843842,0.075861,0.896318,0.020016,...,0.134829,0.256428,0.095440,0.118760,0.059411,0.176775,0.189069,0.307656,0.224084,0.090009
PT-BY9H,0.149988,0.145805,0.849261,0.902912,0.651986,0.834157,0.917063,0.101837,0.941423,0.014101,...,0.145478,0.330483,0.119385,0.091408,0.074204,0.141996,0.226845,0.302488,0.286071,0.051107
TBI-AUTO73307-PT-314I,0.130532,0.122833,0.861136,0.907610,0.748538,0.839395,0.863272,0.071947,0.940628,0.031390,...,0.227004,0.291872,0.120238,0.108485,0.062730,0.167610,0.193875,0.426867,0.272743,0.072089


In [88]:
# map specimen ids
mappable_meth_samples = list(set(meth.index).intersection(biospecimen_metadata.specimenID))
len(mappable_meth_samples)

740

In [89]:
# filter samples then map
meth= meth.loc[mappable_meth_samples]
meth.index = [specimenID_to_individualID_map[name] for name in meth.index]
meth.head()

TargetID,cg00000165,cg00000363,cg00000957,cg00001349,cg00001364,cg00001446,cg00001534,cg00001583,cg00001593,cg00002028,...,ch.22.31817810F,ch.22.33863861F,ch.22.533187F,ch.22.740407F,ch.22.757911F,ch.22.772318F,ch.22.43177094F,ch.22.909671F,ch.22.46830341F,ch.22.1008279F
R7911202,0.189227,0.118234,0.693711,0.908018,0.811888,0.834140,0.875396,0.031570,0.933606,0.015636,...,0.216568,0.292066,0.152181,0.116731,0.107539,0.237625,0.197735,0.467760,0.286330,0.034278
R3553915,0.160225,0.149752,0.833319,0.905652,0.725677,0.888802,0.904158,0.083794,0.939560,0.021319,...,0.210502,0.313304,0.114800,0.109839,0.130294,0.210949,0.221561,0.397870,0.280114,0.054361
R6670564,0.198714,0.147976,0.823878,0.870528,0.748641,0.832278,0.882626,0.099319,0.899963,0.026024,...,0.187526,0.259543,0.121234,0.132565,0.058931,0.148394,0.188438,0.385960,0.224694,0.053840
R7641350,0.221020,0.119004,0.798295,0.934531,0.741652,0.842379,0.882011,0.078228,0.895151,0.026213,...,0.116032,0.282812,0.104178,0.110067,0.079443,0.233106,0.192201,0.372466,0.241334,0.031352
R7369657,0.197162,0.130458,0.824241,0.927992,0.668381,0.832066,0.858846,0.074053,0.914520,0.032506,...,0.108858,0.259654,0.098608,0.077038,0.063095,0.147923,0.176396,0.412925,0.248617,0.060444


In [90]:
# check for missing values
meth.isna().sum().sum()

0

In [91]:
common_ids = list(set(clinical_data.index).intersection(meth.index))
methylation_data = pd.concat([clinical_data.loc[common_ids], meth.loc[common_ids]], axis =1)
methylation_data.index.name = "Sample ID"

In [92]:
microRNA_data
gene_expression_data
Metabolon_data
proteomics_data
methylation_data

,Gender,Race,PMI,Braak,Diagnosis,cg00000165,cg00000363,cg00000957,cg00001349,cg00001364,...,ch.22.31817810F,ch.22.33863861F,ch.22.533187F,ch.22.740407F,ch.22.757911F,ch.22.772318F,ch.22.43177094F,ch.22.909671F,ch.22.46830341F,ch.22.1008279F
Sample ID,,,,,,,,,,,,,,,,,,,,,
R3426726,Female,1,1.583333,4,MCI+,0.174993,0.114520,0.758179,0.890740,0.677411,...,0.138644,0.267636,0.137327,0.113511,0.063697,0.186982,0.175239,0.331336,0.233546,0.058007
R5256488,Female,1,9.083333,4,MCI,0.157094,0.173755,0.841470,0.835279,0.691981,...,0.147719,0.221689,0.092141,0.109841,0.040565,0.100641,0.160724,0.319925,0.215566,0.060442
R9101940,Female,1,2.916667,1,Control,0.212631,0.133191,0.856962,0.896520,0.732280,...,0.161441,0.278719,0.124027,0.126043,0.046589,0.121469,0.202367,0.328242,0.255847,0.034101
R5415701,Female,1,11.250000,0,MCI,0.176137,0.162226,0.853370,0.927061,0.714723,...,0.161154,0.305721,0.117026,0.120890,0.061606,0.182815,0.180115,0.445154,0.259632,0.052687
R3328867,Male,1,8.000000,3,Control,0.116003,0.115539,0.754384,0.900874,0.812925,...,0.183394,0.293722,0.153632,0.151452,0.097790,0.187654,0.217187,0.450310,0.274669,0.058896
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R9225354,Female,1,4.500000,1,MCI,0.144274,0.128822,0.846517,0.911085,0.746814,...,0.232326,0.335959,0.110965,0.099598,0.088960,0.205313,0.246713,0.449742,0.293986,0.059465
R4078277,Male,1,5.183333,2,MCI,0.145351,0.138476,0.745715,0.904071,0.746139,...,0.151896,0.303136,0.113234,0.106143,0.038997,0.102187,0.197035,0.358341,0.282669,0.055789
R5927382,Male,1,4.466667,3,MCI,0.137625,0.129398,0.736581,0.921002,0.729819,...,0.172074,0.327330,0.135367,0.126734,0.047507,0.187321,0.239888,0.430804,0.293839,0.054872


#### Select AD Samples Common Across All Omics and save

In [93]:
valid_classes = ["AD", "Control"]

microRNA_ids = set(microRNA_data[microRNA_data["Diagnosis"].isin(valid_classes)].index)
methylation_ids = set(methylation_data[methylation_data["Diagnosis"].isin(valid_classes)].index)
transcript_ids = set(transcript_expression_data[transcript_expression_data["Diagnosis"].isin(valid_classes)].index)
proteomics_ids = set(proteomics_data[proteomics_data["Diagnosis"].isin(valid_classes)].index)
metabolomics_ids = set(Metabolon_data[Metabolon_data["Diagnosis"].isin(valid_classes)].index)
clinical_ids = set(clinical_data[clinical_data["Diagnosis"].isin(valid_classes)].index)

common_ad_samples = (
    microRNA_ids
    & methylation_ids
    & transcript_ids
    #& proteomics_ids
    & metabolomics_ids
    & clinical_ids
)
len(common_ad_samples)

155

In [94]:
# Save Omics
microRNA_data.loc[common_ad_samples].to_csv("../data/ROSMAP/microRNA.csv")
gene_expression_data.loc[common_ad_samples].to_csv("../data/ROSMAP/gene_expression_data.csv")
Metabolon_data.loc[common_ad_samples].to_csv("../data/ROSMAP/metabolomics_data.csv")
#proteomics_data.loc[common_ad_samples].to_csv("../data/ROSMAP/proteomics_data.csv")
methylation_data.loc[common_ad_samples].to_csv("../data/ROSMAP/methylation_data.csv")